In [ ]:
# Main working folder

# Task 1: File ingestion and regular expressions

import re
from pathlib import Path

# 1. Write a robust file reader function


def read_file(file_path: Path) -> tuple[str, str]:
    """
    Function reads files, and returns a tuple that 
    contains both the content of the file (as strings), as 
    well as the encoding that is used. The function is able to 
    handle both UTF-8 and iso-8859-1 encoding. 
    
    Args: Path to the file that we want to read.
    Returns: text file with content, encoding
    """
    try: 
        text = file_path.read_text(encoding="utf-8")
        return text, "utf-8"
    except UnicodeDecodeError: # If utf-8 fails
        text = file_path.read_text(encoding="iso-8859-1")
        return text, "iso-8859-1"


# 2. Write a log parser function


def parser_func(log_content: str) -> list[dict]:
    """
    Function looks through lines, finding timestamp, register_id, 
    item, price, currency, and status while ignoring
    noice lines. Everything is collected in a dictorinary. 

    Args: string from a log file.
    Returns: a list of dictionaries. Each dictionary contains one valid
    transaction record.  
    """

    # \[ and \] finds square brackets, (?P<name>) gives it the appropriate
    # name, .*? to pull out content from square brackets. 
    valid_lines = (
        r"\[(?P<timestamp>.*?)\] "  
      r"\[(?P<register_id>.*?)\] "  
      r"item=(?P<item>\w+) " # Words with one or more letters/numbers
      r"price=(?P<price>[\d.]+) " # Digits, one or more
      r"(?P<currency>[A-Z]{3}) " # Big letters from A-Z, three letters. "EUR" as currency
      r"\((?P<status>\w+)\)"
    )


    transaction_record = []

    # Looks through the log file and checks lines that match 
    # the outline of valid_lines. Everything else (like noise lines)
    # is ignored. 
    for match in re.finditer(valid_lines, log_content):
        convert_dictionary = match.groupdict() #Dictionary
        convert_dictionary["price"]=float(convert_dictionary["price"])
        transaction_record.append(convert_dictionary)

    return transaction_record



In [ ]:
# Task 2: Recursive utility functions 

import json
from pathlib import Path


# 1. Resursive directory search function
def find_files(directory: str|Path) -> list[Path]:
    """
    Function finds all log files in a directionary. Uses 
    Path.iterdir()to go through items. 
    Path.glob(), Path.rglob(), os.walk() are not used. 

    Args: Folder that we want to look through. 
    Returns: list of Path objects from .log files. 
    """

    directory = Path(directory)

    log_file = []

    if directory.is_dir():
        for item in directory.iterdir(): # Looks through dir
            if item.is_dir(): 
                found_logs = find_files(item) # Looks deeper in folder
                log_file.extend(found_logs)
            elif item.is_file() and item.suffix == ".log": # Adds .log files
                log_file.append(item)

    return log_file


# 2. Find active registers
def active_register_id(data: dict|list) -> set[str]:
    """
    Function goes through dictionaries and lists found in store_config.json. 
    Function finds all registered id strings that are active. 
    
    Args: dataset, dictionary or list, that will be seached.
    Returns: a set of all register id strings that are active. 
    """

    active_register=set()

    if isinstance(data, dict): # If data is a dictionary
        if "id" in data and data.get("active") is True: # If dictionary is register
            active_register.add(str(data["id"]))
        for value in data.values(): # Looks deeper in dictionary
           active_register.update(active_register_id(value))

    elif isinstance(data, list): # If data is a list
        for item in data: # Looks deeper in list
            active_register.update(active_register_id(item))

    return active_register



In [ ]:
# Task 3: Functions for cleaning and analysing data using NumPy

import numpy as np

# 1. Function for removing price outliers 

def price_outliers(price: list[float]|np.ndarray) -> np.ndarray: 

    """
    Function removes price outliers. Price is put into a 1D NumPy array and
    outliers where value deviates from the mean by more than 3 standard deviations
    are removed and replaced with the median of valid prices. 

    Args: A list of NumPy array containing all prices. 
    Returns: numpy array without outliers. 
    """
    price = np.array(price, dtype = float) # Converts into numpy array

    mean = np.mean(price) # Mean
    sd = np.std(price) # Standard deviation

    # If price does not deviate with 3*sd
    valid_prices = []
    for p in price:
        if np.abs(price-mean) <= 3 * sd: 
            valid_prices.append(p)

    median = np.median(valid_prices) # Median of valid prices
    outliers = np.abs(price-mean) > 3 * sd
    price[outliers] = median # Meplace outliers with median

    return price


# 2. Function that computes ratio and addition of epsilon


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Function computes element-wise refund ratio and adds epsilon to prevent zero-division 
    errors. 

    Args: sales_count is numpy array with all the sales per item, and refund_count
    is a numpy array with the refunds per item. 
    Returns: Refund ratio for each item as numpy array. 
    """
    
    epsilon = 1e-6
    ratios = []

    for i in range(len(sales_count)): 
        sale = sales_count[i]
        refund = refund_count[i]

        # Calculate the refund ratio
        ratio = refund / (sale + epsilon)
        ratios.append(ratio)

    return np.array(ratios)



In [11]:
# Task 4: DataFrame manipulation with Pandas

import pandas as pd
import numpy as np

# Task 4.1: Convert transactions to a DataFrame

def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Converts a list of transaction dictionaries into a DataFrame.
    Converts timestamp to datetime, sets it as the index,
    and sorts the transactions chronologically.
    """
    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df


# Task 4.2: Calculate hourly revenue per item

def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups transactions by item and calculates the total price
    in one-hour intervals.
    """
    hourly = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .reset_index()
    )

    return hourly


# Task 4.3: Create a pivot table by item and status

def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table showing total transaction revenue
    grouped by item and status.
    """
    pivot = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

    pivot = pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )

    return pivot


# Task 4.4: Calculate refund ratio per item using Task 3

def refund_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the refund ratio for each item by calling
    compute_refund_ratio() from Task 3.
    """

    # Count transactions for each item and status
    counts = pd.crosstab(df["item"], df["status"])

    # Ensure both columns exist, even if a status is missing
    counts = counts.reindex(
        columns=["SUCCESS", "REFUND"],
        fill_value=0
    )

    # Convert counts to NumPy arrays
    sales_count = counts["SUCCESS"].to_numpy()
    refund_count = counts["REFUND"].to_numpy()

    # Call the function already defined in Task 3
    ratios = compute_refund_ratio(sales_count, refund_count)

    # Create the result DataFrame
    result = pd.DataFrame({
        "item": counts.index,
        "refund_ratio": ratios
    })

    return result


# Testing Task 04

test_transactions = [
    {"timestamp": "2026-01-01 10:15:00", "item": "Coffee", "price": 30.0, "status": "SUCCESS"},
    {"timestamp": "2026-01-01 10:30:00", "item": "Coffee", "price": 30.0, "status": "SUCCESS"},
    {"timestamp": "2026-01-01 11:00:00", "item": "Coffee", "price": 30.0, "status": "REFUND"},
    {"timestamp": "2026-01-01 11:20:00", "item": "Tea", "price": 25.0, "status": "SUCCESS"}
]

test_df = create_dataframe(test_transactions)

print("Task 4.1: DataFrame")
display(test_df)

print("Task 4.2: Hourly revenue")
display(hourly_revenue(test_df))

print("Task 4.3: Status pivot table")
display(create_status_pivot(test_df))

print("Task 4.4: Refund ratio")
display(refund_ratio(test_df))

Task 4.1: DataFrame


,item,price,status
timestamp,,,
2026-01-01 10:15:00,Coffee,30.0,SUCCESS
2026-01-01 10:30:00,Coffee,30.0,SUCCESS
2026-01-01 11:00:00,Coffee,30.0,REFUND
2026-01-01 11:20:00,Tea,25.0,SUCCESS


Task 4.2: Hourly revenue


,item,timestamp,price
0,Coffee,2026-01-01 10:00:00,60.0
1,Coffee,2026-01-01 11:00:00,30.0
2,Tea,2026-01-01 11:00:00,25.0


Task 4.3: Status pivot table


status,SUCCESS,REFUND,CANCELLED
item,,,
Coffee,60.0,30.0,0
Tea,25.0,0.0,0


Task 4.4: Refund ratio


,item,refund_ratio
0,Coffee,0.5
1,Tea,0.0


In [13]:
# Task 5: Create full workflow

from pathlib import Path
import json
import numpy as np
import pandas as pd

# Step 1: Find all log files
data_folder = Path("Data")

log_files = find_files(data_folder)
print("Number of log files:", len(log_files))


# Step 2: Find active register IDs
config_files = list(data_folder.rglob("store_config.json"))

if not config_files:
    raise FileNotFoundError("Could not find store_config.json")

active_registers = set()

for config_file in config_files:
    with open(config_file, encoding="utf-8") as f:
        config_data = json.load(f)

    active_registers.update(active_register_id(config_data))

print("Active registers:", active_registers)


# Step 3: Read and parse all log files
all_transactions = []

for log_file in log_files:
    text, encoding = read_file(log_file)
    transactions = parser_func(text)

    for transaction in transactions:
        if transaction["register_id"] in active_registers:
            all_transactions.append(transaction)

print("Number of transactions:", len(all_transactions))

if not all_transactions:
    raise ValueError("No transactions found for active registers")


# Step 4: Create the initial DataFrame
df = create_dataframe(all_transactions)

print("\nInitial DataFrame:")
display(df.head())


# Step 5: Clean price outliers using Task 3
def price_outliers(prices):
    prices = np.asarray(prices, dtype=float)

    mean = np.mean(prices)
    sd = np.std(prices)

    valid_mask = np.abs(prices - mean) <= 3 * sd
    valid_prices = prices[valid_mask]

    median = np.median(valid_prices)

    cleaned_prices = prices.copy()
    cleaned_prices[~valid_mask] = median

    return cleaned_prices
df["price"] = price_outliers(df["price"].to_numpy())

print("\nDataFrame after outlier cleaning:")
display(df.head())


# Step 6: Correct NOK prices
df.loc[df["currency"] == "NOK", "price"] *= 10.0

print("\nDataFrame after currency correction:")
display(df.head())


# Step 7: Generate reports
hourly_report = hourly_revenue(df)
status_pivot = create_status_pivot(df)
refund_report = refund_ratio(df.reset_index(drop = True))

print("\nHourly revenue:")
display(hourly_report.head())

print("\nStatus pivot table:")
display(status_pivot)

print("\nRefund ratios:")
display(refund_report)


# Step 8: Save output files

output_folder = Path("output")
output_folder.mkdir(exist_ok = True)

# Save cleaned transactions as CSV
df.to_csv(output_folder / "cleaned_transactions.csv")

# Save hourly revenue as JSON
hourly_report.to_json(
    output_folder / "hourly_revenue.json",
    orient="records",
    indent=4,
    date_format="iso"
)

# Save status pivot table as CSV
status_pivot.to_csv(output_folder / "status_pivot_table.csv")

# Save refund ratios as CSV
refund_report.to_csv(
    output_folder / "item_refund_ratios.csv",
    index=False
)


Number of log files: 9
Active registers: {'POS_01', 'POS_02', 'POS_03'}
Number of transactions: 720

Initial DataFrame:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:42,POS_02,Sandwich,7.19,EUR,SUCCESS
2026-10-15 07:01:14,POS_03,Sandwich,7.21,USD,SUCCESS
2026-10-15 07:01:19,POS_02,Latte,4.86,EUR,SUCCESS
2026-10-15 07:01:30,POS_01,Sandwich,7.28,NOK,CANCELLED
2026-10-15 07:01:40,POS_02,Muffin,3.65,EUR,CANCELLED



DataFrame after outlier cleaning:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:42,POS_02,Sandwich,7.19,EUR,SUCCESS
2026-10-15 07:01:14,POS_03,Sandwich,7.21,USD,SUCCESS
2026-10-15 07:01:19,POS_02,Latte,4.86,EUR,SUCCESS
2026-10-15 07:01:30,POS_01,Sandwich,7.28,NOK,CANCELLED
2026-10-15 07:01:40,POS_02,Muffin,3.65,EUR,CANCELLED



DataFrame after currency correction:


...

,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,1124.92
1,Cappuccino,2026-10-15 08:00:00,994.66
2,Cappuccino,2026-10-15 09:00:00,241.00
3,Croissant,2026-10-15 07:00:00,1094.76
4,Croissant,2026-10-15 08:00:00,1073.98



Status pivot table:


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1481.91,505.91,372.76
Croissant,1612.22,378.71,227.21
Espresso,996.32,344.06,136.00
Latte,1213.47,511.56,376.41
Muffin,902.75,316.89,295.88
Sandwich,2470.68,369.03,446.83



Refund ratios:


,item,refund_ratio
0,Cappuccino,0.270588
1,Croissant,0.258065
2,Espresso,0.328947
3,Latte,0.373134
4,Muffin,0.276923
5,Sandwich,0.211765
